    
This is <B>notebook number 1</B> of our <B>NBA Statistics Impact on Salary for Position, Era, and Country</B> that will scrape dataframes to create 4 data files.

The purpose of this notebook is to web scrape all the relevant dataframes required including <B>basic and advanced statistics</B> and <B>countries</B> of players creating <B>Player_Base_Statistics.csv</B>, <B>Player_Advanced_Statistics.csv</B>, and <B>Player_Country.csv</B> and creating a <B>salary</B> dataset from <B>www.sportrec.com</B> called <B>Player_Salary.csv</B>


<H2>Web-Scraping from www.nba.com Basic, Advanced, and Country (Bios) Dataframes </H2>

In [ ]:
# import relevant libraries for dataset 1
import requests
import pandas as pd


In [ ]:
"""The purpose of this cell is to define headers to be used for scraping from www.nba.com and define nba season_list that 
includes season strings from 1996-2023"""

# headers from www.nba.com for scraping
headers = {
    "Connection": "keep-alive",
    "Accept": "application/json, text/plain, */*",
    "x-nba-stats-token": "true",
    "DNT": "1",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/94.0.4606.81 Safari/537.36",
    "x-nba-stats-origin": "stats",
    "Sec-GPC": "1",
    "Origin": "https://www.nba.com",
    "Sec-Fetch-Site": "same-site",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Dest": "empty",
    "Referer": "https://www.nba.com/",
    "Accept-Language": "en-US,en;q=0.9",
}

# season_list for all the NBA years (1996-2023) that will be scraped for Dataset 1 and Dataset 2
season_list = [f"{year}-{str(year + 1)[-2:]}" for year in range(1996, 2023)]


In [ ]:
# create generator of season_list
season_gen = (season for season in season_list)

# function for generating Dataset 1: NBA Base and Advanced Statistics and Dataset 2: Salary


def generate_df(
    seasons=season_gen, stats="Base", url_type="NBA_Statistics", per_mode="Per Game"
):
    """The purpose of this function is to generate primary Dataset 1: Statistics Dataframes
    and Dataset 2: Country Dataframe.

    It accepts the following parameters: 
        1) seasons, which is a generator or list
        2) base or advanced option of statistics, which is a string
        3) the string url_type, which is 'NBA_Statistics' for Dataset 1 or any other key for Dataset 2
        4) per_mode, which is a parameter for the Dataset 1 url.

    The urls are links from www.nba.com"""

    # for loop of seasons
    for season in seasons:

        # use the url for NBA Statistics or NBA players' country of origin
        if url_type == "NBA_Statistics":
            url = (
                "https://stats.nba.com/stats/leaguedashplayerstats?College=&Conference=&Country=&DateFrom=&DateTo=&Division=&DraftPick=&DraftYear=&GameScope=&GameSegment=&Height=&LastNGames=0&LeagueID=00&Location=&MeasureType="
                + stats
                + "&Month=0&OpponentTeamID=0&Outcome=&PORound=0&PaceAdjust=N&PerMode="
                + per_mode
                + "&Period=0&PlayerExperience=&PlayerPosition=&PlusMinus=N&Rank=N&Season="
                + season
                + "&SeasonSegment=&SeasonType=Regular%20Season&ShotClockRange=&StarterBench=&TeamID=0&VsConference=&VsDivision=&Weight="
            )
        else:
            url = (
                "https://stats.nba.com/stats/leaguedashplayerbiostats?College=&Conference=&Country=&DateFrom=&DateTo=&Division=&DraftPick=&DraftYear=&GameScope=&GameSegment=&Height=&LastNGames=0&LeagueID=00&Location=&Month=0&OpponentTeamID=0&Outcome=&PORound=0&PerMode=PerGame&Period=0&PlayerExperience=&PlayerPosition=&Season="
                + season
                + "&SeasonSegment=&SeasonType=Regular%20Season&ShotClockRange=&StarterBench=&TeamID=0&VsConference=&VsDivision=&Weight="
            )

        # request the url and convert it to json format, followed by extraction of columns and data
        req_json = requests.get(url=url, headers=headers).json()
        columns = req_json["resultSets"][0]["headers"]
        data = req_json["resultSets"][0]["rowSet"]

        # convert json data and columns into pandas
        df = pd.DataFrame(data, columns=columns)

        # include season column for each season dataframe
        df["season_id"] = season
        yield df


In [ ]:
"""The purpose of this cell is to use generate_df functions
 to scrape NBA statistics and convert to csv"""

# use function to scrape NBA base_statistics and convert to csv
dfs_base = generate_df()
player_info_base = pd.concat(dfs_base)
player_info_base.to_csv("Player_Base_Statistics.csv")

# use function to scrape NBA advanced_statistics and convert to csv
season_gen = (season for season in season_list)  # generator for season_list
dfs_advanced = generate_df(seasons=season_gen, stats="Advanced")
player_info_advanced = pd.concat(dfs_advanced)
player_info_advanced.to_csv("Player_Advanced_Statistics.csv")


<H2> Web Scraping the Secondary Country Dataset-Dataset 2 from www.nba.com </H2>

In [ ]:
# import relevant libraries for scraping Dataset 2
import requests
import pandas as pd


In [ ]:
# create a generator of season_list
season_gen = (season for season in season_list)

# use generate_df from secondary dataset: a country dataframe
country_df = generate_df(seasons=season_gen, stats="Base", url_type="Country")

# convert country data to csv
player_country = pd.concat(country_df)
player_country.to_csv("Player_Country.csv", index=False)


<H2> Web Scraping Tertiary NBA Player Salary Dataset-Dataset 3 from www.sportrac.com </H2>

In [ ]:
# import relevant libraries for scraping Dataset 3
import pandas as pd


In [ ]:
# function for generating Dataset 3: Salary
def generate_salary_df():
    """The purpose of this function is to generate
    the salary dataframe which is the tertiary dataset. It
    accepts no parameters"""

    # initialize dataframe
    df_salary = pd.DataFrame()

    # all NBA player positions in url link
    positions = [
        "point-guard",
        "shooting-guard",
        "small-forward",
        "power-forward",
        "center",
    ]

    # initialize counter
    count = 0

    # for loop for each player position
    for position in positions:

        # counter to know first dataset
        count += 1

        # url_salary
        url = "https://www.spotrac.com/nba/contracts/sort-value/{}/all-time/limit-2000/".format(
            position
        )

        # scrape table from website
        pos_table = pd.read_html(url)
        pos_df = pd.DataFrame(pos_table[0])

        # get rid of header if not the first table
        if count == 1:
            pass
        else:
            pos_df = pos_df.iloc[1:, :]

        # concatenate each table
        df_salary = pd.concat([df_salary, pos_df], axis=0)

    return df_salary


In [ ]:
# convert salary dataframe to csv file
df_salary = generate_salary_df()
df_salary.to_csv("Player_Salary.csv", index=False)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=0b855501-6f2c-4764-816d-be07593df653' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>